# VISION AI : entraînement sur Colab
Deux usages, **dans cet ordre de rentabilité** :

1. **Partie B (GPU)** : exporter un YOLO plus précis (v8s ou v8m) pour mieux détecter les téléphones. C'est là que le GPU gratuit de Colab sert vraiment.
2. **Partie A (CPU)** : entraîner tes gestes personnalisés avec augmentation + recherche d'hyperparamètres. Colab n'a que 2 vCPU : ce n'est pas plus rapide que ton PC, mais pratique si tu as beaucoup de données.

*Menu Exécution > Modifier le type d'exécution > GPU T4 pour la partie B.*
Aucune image n'est envoyée nulle part : tu n'uploades que des fichiers de landmarks (nombres), jamais de photos.

In [ ]:
!git clone --depth 1 https://github.com/Souraka229/vision-ai /content/vision-ai
%cd /content/vision-ai
!pip -q install scikit-learn joblib numpy

## Partie A : gestes personnalisés
Sur ton PC : `data/gestures/shared/` contient un fichier `.json` par geste. Zippe ce dossier et envoie-le ici.

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil
up = files.upload()  # choisis ton zip de data/gestures/shared
d = pathlib.Path('/content/gestures'); shutil.rmtree(d, ignore_errors=True); d.mkdir()
for name in up:
    zipfile.ZipFile(name).extractall(d)
print(sorted(p.name for p in d.rglob('*.json')))

In [ ]:
# Le premier dossier contenant des .json est utilisé
data = next(p.parent for p in pathlib.Path('/content/gestures').rglob('*.json'))
!python scripts/train_gestures.py --data "{data}" --out /content/custom_shared.joblib --augment 4

In [ ]:
files.download('/content/custom_shared.joblib')
# Place le fichier dans backend/models/custom_shared.joblib sur ton PC, puis relance VISION AI.

## Partie B : YOLO plus précis (GPU)
`yolov8s` est ~3x plus précis sur les petits objets que `yolov8n`, mais ~3x plus lent sur CPU. Teste les deux sur ton PC (People & Objects) et garde le meilleur compromis.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics onnx onnxslim

In [ ]:
from ultralytics import YOLO
MODEL = 'yolov8s'   # ou 'yolov8m' (plus précis, plus lent)
m = YOLO(MODEL + '.pt')
path = m.export(format='onnx', imgsz=640, opset=12, simplify=True, half=False)
print(path)

In [ ]:
import os
from google.colab import files
files.download(f'{MODEL}.onnx')
# Sur ton PC : copie le fichier dans backend/models/ puis, dans config/settings.json, mets
#   "yolo_model": "yolov8s.onnx"   (ou via l'API POST /api/settings)

### Remarques
- Les poids YOLO d'Ultralytics sont sous licence **AGPL-3.0** : ils ne sont pas inclus dans le dépôt, tu les génères toi-même.
- Fine-tuner YOLO sur tes propres images demande un dataset annoté ; le détecteur de l'app attend la tête COCO à 80 classes.